In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression , Lasso ,ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CREATE SYNTHETIC DATA WITH MULTICOLLINEARITY
np.random.seed(42)
n_samples = 15000

# True area signal (what actually matters)
true_area = np.random.normal(2000, 500, n_samples)

# Create 4 highly correlated area features
Area = true_area + np.random.normal(0, 20, n_samples)
LivingArea = true_area + np.random.normal(0, 30, n_samples)
BuiltArea = true_area + np.random.normal(0, 25, n_samples)
CarpetArea = true_area + np.random.normal(0, 35, n_samples)

# Add some other features
Bedrooms = np.random.randint(1, 6, n_samples)
Bathrooms = np.random.randint(1, 4, n_samples)
Age = np.random.randint(0, 50, n_samples)

# Generate price (only truly depends on Area + others)
noise = np.random.normal(0, 50000, n_samples)
price = 200 * true_area + 50000 * Bedrooms + 30000 * Bathrooms - 1000 * Age + noise

# Create DataFrame
df = pd.DataFrame({
    'Area': Area,
    'LivingArea': LivingArea,
    'BuiltArea': BuiltArea,
    'CarpetArea': CarpetArea,
    'Bedrooms': Bedrooms,
    'Bathrooms': Bathrooms,
    'Age': Age,
    'Price': price
})

print("Correlation Matrix (Multicollinearity Check):")
print(df.head(5))
# checking the corr of the similar dataset
print(df[['Area','LivingArea','BuiltArea','CarpetArea']].corr().round(3))


X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)






Correlation Matrix (Multicollinearity Check):
          Area   LivingArea    BuiltArea   CarpetArea  Bedrooms  Bathrooms  \
0  2245.488612  2188.939917  2274.601638  2251.116398         4          1   
1  1930.214731  1899.218280  1911.354530  1950.569261         3          3   
2  2325.130167  2306.233417  2353.829360  2335.782837         5          2   
3  2780.452157  2766.004996  2759.986785  2716.787931         1          3   
4  1867.978966  1913.648182  1904.327142  1876.405602         1          3   

   Age          Price  
0   43  667781.592967  
1   44  662790.264898  
2   15  751016.114938  
3   31  675339.889431  
4    6  499487.865117  
             Area  LivingArea  BuiltArea  CarpetArea
Area        1.000       0.997      0.998       0.997
LivingArea  0.997       1.000      0.997       0.996
BuiltArea   0.998       0.997      1.000       0.996
CarpetArea  0.997       0.996      0.996       1.000


In [13]:
# Grid search over both alpha AND l1_ratio
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# Create a pipeline with StandardScaler and ElasticNet

# 4. Elastic Net (BEST OF BOTH)
elastic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('elastic', ElasticNet(max_iter=10000))
])



elastic_params = {
    'elastic__alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100, 500],
    'elastic__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]
}
elastic_grid = GridSearchCV(elastic_pipeline, elastic_params, cv=5, scoring='r2')
elastic_grid.fit(X_train, y_train)

# Elastic Net
elastic_pred = elastic_grid.predict(X_test)
print(f"\nElastic Net (alpha={elastic_grid.best_params_['elastic__alpha']}, "
      f"l1_ratio={elastic_grid.best_params_['elastic__l1_ratio']:.2f}):")
print(f"  R²: {r2_score(y_test, elastic_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, elastic_pred)):,.0f}")



print("\nRunning GridSearchCV (this might take a moment)...")

# Create pipeline
elastic_pipeline_gs = Pipeline([
    ('scaler', StandardScaler()),
    ('elastic', ElasticNet(max_iter=10000, random_state=42))
])

# Grid search parameters
param_grid_pipeline = {
    'elastic__alpha': [0.001, 0.01, 0.1, 1, 10, 50],
    'elastic__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

grid_search = GridSearchCV(
    elastic_pipeline_gs,
    param_grid_pipeline,
    cv=5,
    scoring='r2',
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

grid_search.fit(X_train, y_train)  # Pipeline handles scaling internally!

print(f"\nBest params (pipeline grid search): {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")





Elastic Net (alpha=0.01, l1_ratio=0.90):
  R²: 0.8679
  RMSE: $50,137

Running GridSearchCV (this might take a moment)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best params (pipeline grid search): {'elastic__alpha': 0.01, 'elastic__l1_ratio': 0.9}
Best CV score: 0.8623
